In [ ]:
import ccxt
import pandas as pd
from datetime import datetime, timedelta

def fetch_ohlcv_data(exchange_id='binance', symbol='BTC/USDT', 
                      timeframe='1h', days_back=1095):
    """
    Fetch historical OHLCV data from exchange
    """
    exchange = ccxt.binance({
        'enableRateLimit': True,
    'timeout': 30000} )

    
    since = exchange.parse8601(
        (datetime.now() - timedelta(days=days_back)).isoformat()
    )
    
    all_ohlcv = []
    
    while since < exchange.milliseconds():
        ohlcv = exchange.fetch_ohlcv(symbol, timeframe, since, limit=1000)
        if not ohlcv:
            break
        since = ohlcv[-1][0] + 1
        all_ohlcv.extend(ohlcv)
        
    df = pd.DataFrame(
        all_ohlcv, 
        columns=['timestamp', 'open', 'high', 'low', 'close', 'volume']
    )
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    df.set_index('timestamp', inplace=True)
    
    return df

# Fetch 3 years of hourly Bitcoin data
btc_data = fetch_ohlcv_data()
print(f"Loaded {len(btc_data)} hourly candles")

Loaded 26277 hourly candles


In [5]:
import numpy as np

def engineer_features(df):
    """
    Create technical features for mean reversion strategy
    """
    # Volatility measures
    df['returns'] = df['close'].pct_change()
    df['volatility_20'] = df['returns'].rolling(20).std()
    df['volatility_spike'] = df['volatility_20'] / df['volatility_20'].rolling(100).mean()
    
    # Volume analysis
    df['volume_ma_20'] = df['volume'].rolling(20).mean()
    df['volume_ratio'] = df['volume'] / df['volume_ma_20']
    
    # Mean reversion indicator
    df['price_ma_50'] = df['close'].rolling(50).mean()
    df['distance_from_ma'] = (df['close'] - df['price_ma_50']) / df['price_ma_50']
    
    # Z-score of returns (standardized deviation)
    df['returns_zscore'] = (
        (df['returns'] - df['returns'].rolling(100).mean()) / 
        df['returns'].rolling(100).std()
    )
    
    # Bollinger Band position
    df['bb_middle'] = df['close'].rolling(20).mean()
    df['bb_std'] = df['close'].rolling(20).std()
    df['bb_upper'] = df['bb_middle'] + (df['bb_std'] * 2)
    df['bb_lower'] = df['bb_middle'] - (df['bb_std'] * 2)
    df['bb_position'] = (df['close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])
    
    return df.dropna()

btc_data = engineer_features(btc_data)

In [3]:
class MeanReversionStrategy:
    def __init__(self, stop_loss=0.03, target_return=0.02, max_hold_hours=48):
        self.stop_loss = stop_loss
        self.target_return = target_return
        self.max_hold_hours = max_hold_hours
        self.position = None
        self.entry_price = None
        self.entry_time = None
        
    def generate_signals(self, df):
        """
        Generate trading signals based on strategy rules
        """
        signals = pd.DataFrame(index=df.index)
        signals['signal'] = 0
        
        # Long entry conditions
        entry_condition = (
            (df['bb_position'] < 0.1) &
            (df['volatility_spike'] > 1.5) &
            (df['volume_ratio'] > 1.3) &
            (df['returns_zscore'] < -2)
        )
        
        signals.loc[entry_condition, 'signal'] = 1
        
        return signals
    
    def backtest(self, df):
        """
        Backtest strategy with realistic execution assumptions
        """
        signals = self.generate_signals(df)
        
        positions = []
        trades = []
        
        for i in range(len(df)):
            current_time = df.index[i]
            current_price = df.iloc[i]['close']
            
            # Check if we have an open position
            if self.position is not None:
                hours_held = (current_time - self.entry_time).total_seconds() / 3600
                pnl_pct = (current_price - self.entry_price) / self.entry_price
                
                # Exit conditions
                exit_signal = False
                exit_reason = None
                
                # Stop loss
                if pnl_pct <= -self.stop_loss:
                    exit_signal = True
                    exit_reason = 'stop_loss'
                
                # Target reached (price returned to MA)
                elif current_price >= df.iloc[i]['price_ma_50']:
                    exit_signal = True
                    exit_reason = 'target_reached'
                
                # Maximum hold period
                elif hours_held >= self.max_hold_hours:
                    exit_signal = True
                    exit_reason = 'max_hold'
                
                if exit_signal:
                    trades.append({
                        'entry_time': self.entry_time,
                        'exit_time': current_time,
                        'entry_price': self.entry_price,
                        'exit_price': current_price,
                        'pnl_pct': pnl_pct,
                        'hold_hours': hours_held,
                        'exit_reason': exit_reason
                    })
                    self.position = None
                    self.entry_price = None
                    self.entry_time = None
            
            # Check for new entry signal (only if not in position)
            elif signals.iloc[i]['signal'] == 1:
                self.position = 'long'
                self.entry_price = current_price
                self.entry_time = current_time
            
            positions.append(1 if self.position else 0)
        
        return pd.DataFrame(trades)

# Run backtest
strategy = MeanReversionStrategy()
trades_df = strategy.backtest(btc_data)

print(f"Total trades: {len(trades_df)}")
print(f"Win rate: {(trades_df['pnl_pct'] > 0).mean():.2%}")
print(f"Average return: {trades_df['pnl_pct'].mean():.2%}")

Total trades: 86
Win rate: 59.30%
Average return: -0.11%


In [5]:
def analyze_performance(trades_df, initial_capital=10000, 
                         trading_fee=0.001, slippage=0.0005):
    """
    Calculate realistic returns accounting for costs
    """
    # Apply transaction costs
    trades_df['gross_pnl_pct'] = trades_df['pnl_pct']
    trades_df['fees'] = trading_fee * 2  # Entry + exit
    trades_df['slippage_cost'] = slippage * 2
    trades_df['net_pnl_pct'] = (
        trades_df['gross_pnl_pct'] - 
        trades_df['fees'] - 
        trades_df['slippage_cost']
    )
    
    # Calculate metrics
    metrics = {
        'total_trades': len(trades_df),
        'win_rate': (trades_df['net_pnl_pct'] > 0).mean(),
        'avg_return': trades_df['net_pnl_pct'].mean(),
        'median_return': trades_df['net_pnl_pct'].median(),
        'best_trade': trades_df['net_pnl_pct'].max(),
        'worst_trade': trades_df['net_pnl_pct'].min(),
        'sharpe_ratio': trades_df['net_pnl_pct'].mean() / trades_df['net_pnl_pct'].std() if len(trades_df) > 0 else 0,
        'profit_factor': abs(trades_df[trades_df['net_pnl_pct'] > 0]['net_pnl_pct'].sum() / 
                            trades_df[trades_df['net_pnl_pct'] < 0]['net_pnl_pct'].sum()) if (trades_df['net_pnl_pct'] < 0).any() else float('inf')
    }
    
    # Portfolio simulation
    portfolio_value = initial_capital
    equity_curve = [portfolio_value]
    
    for pnl in trades_df['net_pnl_pct']:
        portfolio_value *= (1 + pnl)
        equity_curve.append(portfolio_value)
    
    # Drawdown analysis
    equity_series = pd.Series(equity_curve)
    rolling_max = equity_series.expanding().max()
    drawdown = (equity_series - rolling_max) / rolling_max
    
    metrics['max_drawdown'] = drawdown.min()
    metrics['final_capital'] = portfolio_value
    metrics['total_return'] = (portfolio_value - initial_capital) / initial_capital
    
    return metrics, equity_curve

metrics, equity_curve = analyze_performance(trades_df)

print("\n=== Strategy Performance ===")
for key, value in metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")


=== Strategy Performance ===
total_trades: 86
win_rate: 0.5581
avg_return: -0.0041
median_return: 0.0023
best_trade: 0.0744
worst_trade: -0.0654
sharpe_ratio: -0.1369
profit_factor: 0.7079
max_drawdown: -0.3932
final_capital: 6776.1419
total_return: -0.3224


In [6]:
def analyze_performance(trades_df, initial_capital=10000, 
                         trading_fee=0.001, slippage=0.0005):
    """
    Calculate realistic returns accounting for costs
    """
    # Apply transaction costs
    trades_df['gross_pnl_pct'] = trades_df['pnl_pct']
    trades_df['fees'] = trading_fee * 2  # Entry + exit
    trades_df['slippage_cost'] = slippage * 2
    trades_df['net_pnl_pct'] = (
        trades_df['gross_pnl_pct'] - 
        trades_df['fees'] - 
        trades_df['slippage_cost']
    )
    
    # Calculate metrics
    metrics = {
        'total_trades': len(trades_df),
        'win_rate': (trades_df['net_pnl_pct'] > 0).mean(),
        'avg_return': trades_df['net_pnl_pct'].mean(),
        'median_return': trades_df['net_pnl_pct'].median(),
        'best_trade': trades_df['net_pnl_pct'].max(),
        'worst_trade': trades_df['net_pnl_pct'].min(),
        'sharpe_ratio': trades_df['net_pnl_pct'].mean() / trades_df['net_pnl_pct'].std() if len(trades_df) > 0 else 0,
        'profit_factor': abs(trades_df[trades_df['net_pnl_pct'] > 0]['net_pnl_pct'].sum() / 
                            trades_df[trades_df['net_pnl_pct'] < 0]['net_pnl_pct'].sum()) if (trades_df['net_pnl_pct'] < 0).any() else float('inf')
    }
    
    # Portfolio simulation
    portfolio_value = initial_capital
    equity_curve = [portfolio_value]
    
    for pnl in trades_df['net_pnl_pct']:
        portfolio_value *= (1 + pnl)
        equity_curve.append(portfolio_value)
    
    # Drawdown analysis
    equity_series = pd.Series(equity_curve)
    rolling_max = equity_series.expanding().max()
    drawdown = (equity_series - rolling_max) / rolling_max
    
    metrics['max_drawdown'] = drawdown.min()
    metrics['final_capital'] = portfolio_value
    metrics['total_return'] = (portfolio_value - initial_capital) / initial_capital
    
    return metrics, equity_curve

metrics, equity_curve = analyze_performance(trades_df)

print("\n=== Strategy Performance ===")
for key, value in metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")


=== Strategy Performance ===
total_trades: 86
win_rate: 0.5581
avg_return: -0.0041
median_return: 0.0023
best_trade: 0.0744
worst_trade: -0.0654
sharpe_ratio: -0.1369
profit_factor: 0.7079
max_drawdown: -0.3932
final_capital: 6776.1419
total_return: -0.3224


In [7]:
def walk_forward_analysis(df, train_period_months=18, test_period_months=6):
    """
    Walk-forward optimization to prevent overfitting
    """
    results = []
    
    # Split data into train/test windows
    total_months = len(df) // (30 * 24)  # Approximate months
    num_windows = (total_months - train_period_months) // test_period_months
    
    for i in range(num_windows):
        train_start_idx = i * test_period_months * 30 * 24
        train_end_idx = train_start_idx + train_period_months * 30 * 24
        test_end_idx = train_end_idx + test_period_months * 30 * 24
        
        train_data = df.iloc[train_start_idx:train_end_idx]
        test_data = df.iloc[train_end_idx:test_end_idx]
        
        # Train strategy on training period
        strategy = MeanReversionStrategy()
        
        # Test on out-of-sample period
        test_trades = strategy.backtest(test_data)
        
        if len(test_trades) > 0:
            test_metrics, _ = analyze_performance(test_trades)
            results.append({
                'window': i,
                'test_period_start': test_data.index[0],
                'test_period_end': test_data.index[-1],
                'trades': len(test_trades),
                'win_rate': test_metrics['win_rate'],
                'avg_return': test_metrics['avg_return'],
                'sharpe': test_metrics['sharpe_ratio']
            })
    
    return pd.DataFrame(results)

# Perform walk-forward analysis
wf_results = walk_forward_analysis(btc_data)
print("\n=== Walk-Forward Analysis ===")
print(wf_results)
print(f"\nConsistent win rate: {wf_results['win_rate'].mean():.2%}")
print(f"Win rate std dev: {wf_results['win_rate'].std():.2%}")


=== Walk-Forward Analysis ===
   window   test_period_start     test_period_end  trades  win_rate  \
0       0 2025-02-19 14:00:00 2025-08-18 13:00:00      11  0.636364   
1       1 2025-08-18 14:00:00 2026-02-14 13:00:00      15  0.466667   
2       2 2026-02-14 14:00:00 2026-08-13 13:00:00      15  0.666667   

   avg_return    sharpe  
0   -0.000961 -0.027519  
1   -0.010552 -0.453447  
2    0.001052  0.040463  

Consistent win rate: 58.99%
Win rate std dev: 10.78%
